# 01 - Feature Engineering (Final)

**Goal:** Create reproducible engineered features for downstream modeling.

**Inputs:** Previous stage artifacts and configured data paths

**Outputs:** Versioned artifacts for the next stage


## Report-Ready Runbook
1. Update path/config variables
2. Run cells top-to-bottom
3. Capture final metrics/artifacts for README
4. Keep exploratory diagnostics in `notebooks/archive/` only


In [ ]:
#imports
from pyspark.sql.functions import col, isnan, when, count, to_date, to_timestamp
from pyspark.sql.functions import min as spark_min, max as spark_max, sum as spark_sum
from pyspark.sql.types import IntegerType, FloatType, DoubleType, StringType, DateType, TimestampType
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.sql.functions import col, hour, count, to_date, abs, avg, sum, unix_timestamp, when, hour, format_string
from graphframes import GraphFrame
from pyspark.sql.functions import col, hour, count, to_date, abs
from pyspark.sql.functions import unix_timestamp, from_unixtime, to_timestamp
from pyspark.sql.window import Window

In [ ]:
#Feature Engineering final function 

def join_holidays(flights_df, us_holidays):
    us_holidays_df  = us_holidays.withColumnRenamed("Date","Holiday_Date")

    #Join with holidays dataset
    flights_holidays = flights_df.join(us_holidays_df, flights_df["FL_DATE"] == us_holidays_df["Holiday_Date"], "left")
    flights_holidays = flights_holidays.withColumn(
        'Holiday_Encoded',when(col('Holiday').isNotNull(), 1).otherwise(0))
    return flights_holidays

#Create new column for Time of Day categories
def time_of_day(flights_df):
    #extract the hour of the scheduled departure time

    flights_df = flights_df.withColumn('DEP_TIME_HH', hour(flights_df['sched_depart_date_time_UTC']))


    flights_df = flights_df.withColumn('Time_of_Day',
    when((col('DEP_TIME_HH') >= 5) & (col('DEP_TIME_HH') < 12), 'Morning')
    .when((col('DEP_TIME_HH') >= 12) & (col('DEP_TIME_HH') < 17), 'Afternoon')
    .when((col('DEP_TIME_HH') >= 17) & (col('DEP_TIME_HH') < 21), 'Evening')
    .otherwise('Night'))
    return flights_df

# Page Rank function
def pagerank_score(df):
    # Create airport nodes
    airport_nodes = df.select(col("ORIGIN_AIRPORT_ID").alias("id")).distinct()

    # Create flight edges between origin and destination airports
    airport_edges = df.select(col("ORIGIN_AIRPORT_ID").alias("src"), col("DEST_AIRPORT_ID").alias("dst"))

    # Build the graph (Airports as Nodes, Flights as Edges)
    airport_graph = GraphFrame(airport_nodes, airport_edges)

    # Compute PageRank
    pagerank_results = airport_graph.pageRank(resetProbability=0.1, maxIter=20)

    # Extract PageRank scores for each airport
    airport_pagerank = pagerank_results.vertices.select("id", "pagerank")

    # Join PageRank scores for each of the origin and destination airports in the OTPW dataset 
    # flights_with_pagerank = df \
    #     .join(airport_pagerank.withColumnRenamed("id", "ORIGIN_AIRPORT_ID"),"ORIGIN_AIRPORT_ID", "left") \
    #     .withColumnRenamed("pagerank", "origin_pagerank") \
    #     .join(airport_pagerank.withColumnRenamed("id", "DEST_AIRPORT_ID"), "DEST_AIRPORT_ID", "left") \
    #     .withColumnRenamed("pagerank", "dest_pagerank")

    flights_with_pagerank = df \
    .join(airport_pagerank.withColumnRenamed("id", "ORIGIN_AIRPORT_ID"),"ORIGIN_AIRPORT_ID", "left") \
    .withColumnRenamed("pagerank", "origin_pagerank") 
    
    return flights_with_pagerank


def preprocess_data(flights_df):
    #Convert DEP_DELAY_NEW to int
    flights_df = flights_df.withColumn(
        "DEP_DELAY_NEW_int", 
        col("DEP_DELAY_NEW").cast("int")
    )

    #create timestamp fields for window calculations 
    flights_df = flights_df.withColumn("dep_timestamp", col("sched_depart_date_time_UTC").cast("timestamp").cast("long")) \
                 .withColumn("two_hours_prior_depart_timestamp", col("two_hours_prior_depart_UTC").cast("timestamp").cast("long")) 

    return flights_df




#Average delay by airport by time of day by flight date; # Time of Day Window - Average delay by airport by time of day by flight date
def avg_delay_by_airport_time_of_day(flights_df):
    # Define the window specification
    window_spec_daytime = Window.partitionBy("ORIGIN_AIRPORT_ID", "Time_of_Day").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin, time of day, and date
    flights_df = flights_df.withColumn(
        "avg_delay_by_airport_time_of_day",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_daytime)
    )
    return flights_df


# Average delay by airport
def avg_delay_by_airport(flights_df):

    # Define the window specification
    window_spec_airport = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin, time of day, and date
    flights_df = flights_df.withColumn(
        "avg_delay_by_airport",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airport)
    )
    return flights_df

# Average delay by airport by airline
def avg_delay_by_airport_airline(flights_df):
    # Define the window specification
    window_spec_airline = Window.partitionBy("ORIGIN_AIRPORT_ID", "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_byairport_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )
    return flights_df

# Average delay by airline
def avg_delay_by_airline(flights_df):
    # Define the window specification
    window_spec_airline = Window.partitionBy( "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_by_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )
    return flights_df
    
def avg_delay_metrics(flights_df):
    #preprocess data
    #Convert DEP_DELAY_NEW to int
    flights_df = flights_df.withColumn(
        "DEP_DELAY_NEW_int", 
        col("DEP_DELAY_NEW").cast("int")
    )

    flights_df = flights_df.withColumn('DEP_TIME_HH', hour(flights_df['sched_depart_date_time_UTC']))

    #create timestamp fields for window calculations 
    flights_df = flights_df.withColumn("dep_timestamp", col("sched_depart_date_time_UTC").cast("timestamp").cast("long")) \
                 .withColumn("two_hours_prior_depart_timestamp", col("two_hours_prior_depart_UTC").cast("timestamp").cast("long")) 

    
    flights_df = flights_df.withColumn('Time_of_Day',
    when((col('DEP_TIME_HH') >= 5) & (col('DEP_TIME_HH') < 12), 'Morning')
    .when((col('DEP_TIME_HH') >= 12) & (col('DEP_TIME_HH') < 17), 'Afternoon')
    .when((col('DEP_TIME_HH') >= 17) & (col('DEP_TIME_HH') < 21), 'Evening')
    .otherwise('Night'))

    # Average delay by airport
    # Define the window specification
    window_spec_airport = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin
    flights_df = flights_df.withColumn(
        "avg_delay_by_airport",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airport)
    )

    # Average delay by airport by airline
    # Define the window specification
    window_spec_airline = Window.partitionBy("ORIGIN_AIRPORT_ID", "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_byairport_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )


    # Average delay by airline
    # Define the window specification
    window_spec_airline = Window.partitionBy( "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_by_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )

    return flights_df



def freq_metrics(df):
    # df = df.withColumn("dep_timestamp", col("sched_depart_date_time_UTC").cast("timestamp").cast("long")) \
    #                         .withColumn("two_hours_prior_depart_timestamp", col("two_hours_prior_depart_UTC").cast("timestamp").cast("long")) 


    # Define a 24-hour rolling window before `query_time`
    time_window = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp") \
                        .rangeBetween(-86400, -1)  # 24 hours before `query_time`, exclude current flight

    # Count # of flights within the 24-hour window
    df = df.withColumn("num_severe_delayed_flights", sum(col("DEP_DEL15")).over(time_window))\
            .withColumn("num_outgoing_flights", count("*").over(time_window))

    return df

#Call feature engineering main function
def feature_engineering_main(flights_df, us_holidays_df, folder_path):

    flights_df = join_holidays(flights_df, us_holidays_df)

    flights_df = flights_df.cache()
    flights_df = avg_delay_metrics(flights_df)
    flights_df = flights_df.cache()
   
    flights_df = freq_metrics(flights_df)
    flights_df = flights_df.cache()

    flights_df = pagerank_score(flights_df)
    flights_df = flights_df.cache()

    return flights_df



In [ ]:
#feat eng old code
#read input files
folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_otpw_12m_clean.parquet"
df_flights_12m = spark.read.parquet(file_path)

us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")
# 2 min 17 sec
df_flights_12m_feat_new_pagerank_old = feature_engineering_main(df_flights_12m, us_holiday_df, folder_path)

In [ ]:

#read input files
folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_otpw_12m_clean.parquet"
df_flights_12m = spark.read.parquet(file_path)

us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")
#33 sec
df_flights_12m_feat_new_pagerank = feature_engineering_main(df_flights_12m, us_holiday_df, folder_path)


In [ ]:
# Save the DataFrame to the specified path with overwrite mode
folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_flights_12m_feat_new.parquet"
df_flights_12m_feat_new_pagerank.write.mode("overwrite").parquet(file_path)

In [ ]:
#using delta
#Feature Engineering final function 

def join_holidays(flights_df, us_holidays):
    us_holidays_df  = us_holidays.withColumnRenamed("Date","Holiday_Date")

    #Join with holidays dataset
    flights_holidays = flights_df.join(us_holidays_df, flights_df["FL_DATE"] == us_holidays_df["Holiday_Date"], "left")
    flights_holidays = flights_holidays.withColumn(
        'Holiday_Encoded',when(col('Holiday').isNotNull(), 1).otherwise(0))
    return flights_holidays

#Create new column for Time of Day categories
def time_of_day(flights_df):
    #extract the hour of the scheduled departure time

    flights_df = flights_df.withColumn('DEP_TIME_HH', hour(flights_df['sched_depart_date_time_UTC']))


    flights_df = flights_df.withColumn('Time_of_Day',
    when((col('DEP_TIME_HH') >= 5) & (col('DEP_TIME_HH') < 12), 'Morning')
    .when((col('DEP_TIME_HH') >= 12) & (col('DEP_TIME_HH') < 17), 'Afternoon')
    .when((col('DEP_TIME_HH') >= 17) & (col('DEP_TIME_HH') < 21), 'Evening')
    .otherwise('Night'))
    return flights_df

def stage_1_features(flights_df, us_holidays_df, output_path):
    flights_df = join_holidays(flights_df, us_holidays_df)

    #preprocess data
    #Convert DEP_DELAY_NEW to int
    flights_df = flights_df.withColumn("DEP_DELAY_NEW_int", col("DEP_DELAY_NEW").cast("int"))

    flights_df = flights_df.withColumn('DEP_TIME_HH', hour(flights_df['sched_depart_date_time_UTC']))

    #create timestamp fields for window calculations 
    flights_df = flights_df.withColumn("dep_timestamp", col("sched_depart_date_time_UTC").cast("timestamp").cast("long")) \
                 .withColumn("two_hours_prior_depart_timestamp", col("two_hours_prior_depart_UTC").cast("timestamp").cast("long")) 

    
    flights_df = flights_df.withColumn('Time_of_Day',
    when((col('DEP_TIME_HH') >= 5) & (col('DEP_TIME_HH') < 12), 'Morning')
    .when((col('DEP_TIME_HH') >= 12) & (col('DEP_TIME_HH') < 17), 'Afternoon')
    .when((col('DEP_TIME_HH') >= 17) & (col('DEP_TIME_HH') < 21), 'Evening')
    .otherwise('Night'))
    flights_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(output_path)

    return flights_df



def preprocess_data(flights_df):
    #Convert DEP_DELAY_NEW to int
    flights_df = flights_df.withColumn("DEP_DELAY_NEW_int", col("DEP_DELAY_NEW").cast("int"))

    #create timestamp fields for window calculations 
    flights_df = flights_df.withColumn("dep_timestamp", col("sched_depart_date_time_UTC").cast("timestamp").cast("long")) \
                 .withColumn("two_hours_prior_depart_timestamp", col("two_hours_prior_depart_UTC").cast("timestamp").cast("long")) 

    return flights_df



# Average delay by airport
def avg_delay_by_airport(flights_df):

    # Define the window specification
    window_spec_airport = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin, time of day, and date
    flights_df = flights_df.withColumn(
        "avg_delay_by_airport",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airport)
    )
    return flights_df

# Average delay by airport by airline
def avg_delay_by_airport_airline(flights_df):
    # Define the window specification
    window_spec_airline = Window.partitionBy("ORIGIN_AIRPORT_ID", "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_byairport_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )
    return flights_df

# Average delay by airline
def avg_delay_by_airline(flights_df):
    # Define the window specification
    window_spec_airline = Window.partitionBy( "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_by_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )
    return flights_df
    
def stage_2_avg_delay_metrics(flights_df, output_path):

    # Average delay by airport
    # Define the window specification
    window_spec_airport = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin
    flights_df = flights_df.withColumn(
        "avg_delay_by_airport",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airport)
    )

    # Average delay by airport by airline
    # Define the window specification
    window_spec_airline = Window.partitionBy("ORIGIN_AIRPORT_ID", "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_byairport_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )


    # Average delay by airline
    # Define the window specification
    window_spec_airline = Window.partitionBy( "OP_CARRIER").orderBy("two_hours_prior_depart_timestamp").rangeBetween(-86400, -1)

    # Calculate the average delay by airport origin and airlines
    flights_df = flights_df.withColumn(
        "avg_delay_by_airline",
        avg(col("DEP_DELAY_NEW_int")).over(window_spec_airline)
    )

    flights_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(output_path)

    return flights_df



def freq_metrics(flights_df):
    # Define a 24-hour rolling window before `query_time`
    time_window = Window.partitionBy("ORIGIN_AIRPORT_ID").orderBy("two_hours_prior_depart_timestamp") \
                        .rangeBetween(-86400, -1)  # 24 hours before `query_time`, exclude current flight

    # Count # of flights within the 24-hour window
    flights_df = flights_df.withColumn("num_severe_delayed_flights", sum(col("DEP_DEL15")).over(time_window))\
            .withColumn("num_outgoing_flights", count("*").over(time_window))

    return flights_df


def stage_3_freq_metrics(flights_df, output_path):
    flights_df = freq_metrics(flights_df)
    flights_df.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(output_path)
    return flights_df

# Page Rank function
def pagerank_score(df):
    # Create airport nodes
    airport_nodes = df.select(col("ORIGIN_AIRPORT_ID").alias("id")).distinct()

    # Create flight edges between origin and destination airports
    airport_edges = df.select(col("ORIGIN_AIRPORT_ID").alias("src"), col("DEST_AIRPORT_ID").alias("dst"))

    # Build the graph (Airports as Nodes, Flights as Edges)
    airport_graph = GraphFrame(airport_nodes, airport_edges)

    # Compute PageRank
    pagerank_results = airport_graph.pageRank(resetProbability=0.1, maxIter=20)

    # Extract PageRank scores for each airport
    airport_pagerank = pagerank_results.vertices.select("id", "pagerank")

    # Join PageRank scores for each of the origin and destination airports in the OTPW dataset 
    # flights_with_pagerank = df \
    #     .join(airport_pagerank.withColumnRenamed("id", "ORIGIN_AIRPORT_ID"),"ORIGIN_AIRPORT_ID", "left") \
    #     .withColumnRenamed("pagerank", "origin_pagerank") \
    #     .join(airport_pagerank.withColumnRenamed("id", "DEST_AIRPORT_ID"), "DEST_AIRPORT_ID", "left") \
    #     .withColumnRenamed("pagerank", "dest_pagerank")

    flights_with_pagerank = df \
    .join(airport_pagerank.withColumnRenamed("id", "ORIGIN_AIRPORT_ID"),"ORIGIN_AIRPORT_ID", "left") \
    .withColumnRenamed("pagerank", "origin_pagerank") 
    
    return flights_with_pagerank

def stage_4_pageRank(df, output_path):
    df = pagerank_score(df)
    df.write.mode("overwrite").format("delta").option("mergeSchema", "true").save(output_path)
    return df

def feature_engineering_pipeline_main(flights_df, us_holidays_df, base_output_path):
    df_stage1 = stage_1_features(flights_df, us_holidays_df, f"{base_output_path}/stage1")
    df_stage1 = spark.read.format("delta").load(f"{base_output_path}/stage1")
    print("Finished stage 1")

    df_stage2 = stage_2_avg_delay_metrics(df_stage1, f"{base_output_path}/stage2")
    df_stage2 = spark.read.format("delta").load(f"{base_output_path}/stage2")
    print("Finished stage 2")

    df_stage3 = stage_3_freq_metrics(df_stage2, f"{base_output_path}/stage3")
    df_stage3 = spark.read.format("delta").load(f"{base_output_path}/stage3")
    print("Finished stage 3")

    df_final = stage_4_pageRank(df_stage3, f"{base_output_path}/final")
    df_final = spark.read.format("delta").load(f"{base_output_path}/final")
    print("Finished stage 4")
    return df_final

# #read input files
# folder_path = "dbfs:/student-groups/Group_03_03"
# file_path = f"{folder_path}/df_otpw_12m_clean.parquet"
# df_flights_12m = spark.read.parquet(file_path)

# us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")
# # 2 min 26sec
# df_flights_12m_feat_new_pagerank = feature_engineering_pipeline_main(df_flights_12m, us_holiday_df, folder_path)

#usign hybrid delta and caching
#read input files
# folder_path = "dbfs:/student-groups/Group_03_03"
# file_path = f"{folder_path}/df_otpw_12m_clean.parquet"
# df_flights_12m = spark.read.parquet(file_path)

# us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")
# # 3m 12 sec
#df_flights_12m_feat_new_pagerank = feature_engineering_pipeline_main(df_flights_12m, us_holiday_df, folder_path)

#using read delta files (no caching)
#read input files
# folder_path = "dbfs:/student-groups/Group_03_03"
# file_path = f"{folder_path}/df_otpw_12m_clean.parquet"
# df_flights_12m = spark.read.parquet(file_path)

# us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")
# # 2m 39 sec
# df_flights_12m_feat_new_pagerank = feature_engineering_pipeline_main(df_flights_12m, us_holiday_df, folder_path)

In [ ]:
# #read input files
# folder_path = "dbfs:/student-groups/Group_03_03"
# file_path = f"{folder_path}/df_otpw_60m_clean.parquet"
# df_flights_60m = spark.read.parquet(file_path)

# us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")

# df_flights_60m_feat_new = feature_engineering_pipeline_main(df_flights_60m, us_holiday_df, folder_path)

In [ ]:

#read input files
folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_otpw_60m_clean.parquet"
df_flights_60m = spark.read.parquet(file_path)  
"dbfs:/student-groups/Group_03_03/df_otpw_60m_clean.parquet"

us_holiday_df= spark.read.csv("/FileStore/tables/dbfs:/student-groups/Group_03_03/US_Holiday_Dates__2004_2021_-1.csv", header=True, inferSchema=True).select("Date","Holiday")

df_flights_60m_feat_new = feature_engineering_pipeline_main(df_flights_60m, us_holiday_df, folder_path)

In [ ]:
# Save the DataFrame to the specified path with overwrite mode
folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_flights_60m_feat_new.parquet"

df_flights_60m_feat_new.write.mode("overwrite").parquet(file_path)

folder_path = "dbfs:/student-groups/Group_03_03"
file_path = f"{folder_path}/df_otpw_60m_clean.parquet"
df_flights_60m = spark.read.parquet(file_path)  
"dbfs:/student-groups/Group_03_03/df_otpw_60m_clean.parquet"

## Conclusion
Document key outcomes from this stage (dataset shape, selected features, best params, or metrics) before committing.
